# TRICARE 파인튜닝 데이터셋 생성 v2

## 개요
- RAG 체인(`make_rag_chain`)을 활용해 질문-답변 쌍 생성
- 목표: **400개** JSONL 데이터셋
- 파인튜닝 목적: 출처 형식 고정 / 면책 고지 자동 삽입 / 추천 거절 / 개인정보(식별 가능한 경우만) 차단

## 카테고리 비율 (회의 결정 사항)
| 카테고리 | 비율 | 수량(400기준) | 설명 |
|---|---|---|---|
| 면책 고지 | 35% | ~140개 | 일반 보험 정보 제공 + 면책 고지 포함 |
| 추천 방지 | 20% | ~80개 | 플랜 추천 요청 → 거절 |
| 문맥 | 10% | ~40개 | 복잡한 질문, 문맥 파악 필요 |
| 개인정보 차단 | 35% | ~140개 | 식별 가능 개인정보 포함 시만 차단 |

## 개인정보 차단 기준 (회의 결정)
- 차단 X: `"내가 우울증인데 보험 받을 수 있어?"` → 진단명만 있는 경우 일반 답변
- 차단 O: `"제 보험번호 235-463이고 F32 진단인데"` → 개인 식별 정보(보험번호·SSN 등) 포함 시 차단

## 출처 표기 형식 (회의 결정)
- PDF: `[출처]: 파일명(괄호 제거) 연도 p.N`
  - ex. `NGR_HB(국가방위군...) → NGR_HB`
  - ex. `QLEs_FS (2) → QLEs_FS`
- CSV: `[출처]: TRICARE https://tricare.mil/`


---
## Cell 0 · 환경 재로드 (별도 세션에서 실행 시)
- v3 노트북과 같은 커널이면 스킵
- 별도 세션이면 여기서 vector_store 로드 + 함수 재정의

In [1]:
# 별도 세션에서 실행할 때만 이 셀 실행
# v3 노트북과 같은 커널에서 실행 중이면 스킵

import os
from dotenv import load_dotenv
load_dotenv()

PERSIST_TEXT    = './chroma_db'
PERSIST_TABLE   = './chroma_db2'
COLLECTION_TEXT  = 'tricare_rag'
COLLECTION_TABLE = 'tricare_cost_tables'

from langchain_huggingface import HuggingFaceEmbeddings

# BGE-M3 로컬 캐시 경로 직접 지정 (BAAI/bge-m3 로드 실패 시)
# 아래 경로에서 실제 해시 폴더명 확인 후 교체:
# import os; print(os.listdir(r'C:\Users\kjw70\.cache\huggingface\hub\models--BAAI--bge-m3\snapshots'))
embedding_model = HuggingFaceEmbeddings(
    model_name=r'C:\Users\kjw70\.cache\huggingface\hub\models--BAAI--bge-m3\snapshots\5617a9f61b028005a4858fdac845db406aefb181',
    model_kwargs={'device': 'cpu'}
)

from langchain_chroma import Chroma
vector_store = Chroma(
    collection_name=COLLECTION_TEXT,
    embedding_function=embedding_model,
    persist_directory=PERSIST_TEXT
)
table_vector_store = Chroma(
    collection_name=COLLECTION_TABLE,
    embedding_function=embedding_model,
    persist_directory=PERSIST_TABLE
)
print(f'텍스트 벡터 수: {vector_store._collection.count()}')
print(f'표 벡터 수: {table_vector_store._collection.count()}')


텍스트 벡터 수: 0
표 벡터 수: 0


In [2]:
# Cell 0-B: 함수 재정의 (별도 세션 전용)

import re, json as _json
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

model = ChatOpenAI(model='gpt-4o-mini', temperature=0.1)

retriever = vector_store.as_retriever(
    search_type='mmr',
    search_kwargs={'k': 6, 'fetch_k': 20}
)
table_retriever = table_vector_store.as_retriever(
    search_type='mmr',
    search_kwargs={'k': 3, 'fetch_k': 10}
)

def detect_language(text: str) -> str:
    if any('\u4e00' <= c <= '\u9fff' for c in text): return 'zh'
    if any('\u3040' <= c <= '\u30ff' for c in text): return 'ja'
    if any('\uac00' <= c <= '\ud7a3' for c in text): return 'ko'
    return 'en'

LANGUAGE_NAME_MAP = {
    'ko': 'Korean', 'en': 'English', 'zh': 'Chinese',
    'ja': 'Japanese', 'es': 'Spanish', 'fr': 'French',
    'de': 'German', 'other': "the same language as the user's question"
}

def clean_source_name(filename: str) -> str:
    """출처 파일명 정제:
    - 괄호와 괄호 안 내용 제거: NGR_HB(국가방위군...) -> NGR_HB
    - 대괄호와 대괄호 안 숫자 제거: QLEs_FS (2) -> QLEs_FS
    - 확장자 제거
    """
    name = filename
    name = re.sub(r'\s*\([^)]*\)', '', name)   # 소괄호 + 내용 제거
    name = re.sub(r'\s*\[[^\]]*\]', '', name)   # 대괄호 + 내용 제거
    name = re.sub(r'\.(pdf|csv|PDF|CSV)$', '', name)  # 확장자 제거
    return name.strip()

def format_docs(docs):
    """검색된 문서를 출처 포함 텍스트로 변환
    출처 형식: [출처]: 파일명 연도 p.N  또는  [출처]: TRICARE https://tricare.mil/
    """
    result = []
    for doc in docs:
        raw_source = doc.metadata.get('source_file', doc.metadata.get('source', 'unknown'))
        page = doc.metadata.get('page', '')
        content = doc.page_content
        if '[search_tags]' in content:
            content = content.split('[search_tags]')[0].strip()

        # CSV 파일은 공식 사이트 링크로 출처 표기
        if raw_source.endswith('.csv'):
            label = '[출처]: TRICARE https://tricare.mil/'
        else:
            clean_name = clean_source_name(raw_source)
            page_str = f' p.{page}' if page != '' else ''
            label = f'[출처]: {clean_name}{page_str}'

        result.append(f'{label}\n{content}')
    return '\n\n'.join(result)

def make_rag_chain(question: str, ret, model):
    """MMR retriever 기반 RAG 체인 — 데이터셋 생성 전용"""
    language_code = detect_language(question)
    answer_language = LANGUAGE_NAME_MAP.get(language_code, 'English')
    PROMPT_TEMPLATE = (
        'You are a TRICARE health benefits specialist.\n'
        'This system is designed for OCONUS beneficiaries, '
        'primarily Korean residents and USFK (주한미군) personnel.\n'
        'IMPORTANT OCONUS RULES:\n'
        '- In South Korea, TRICARE is the PRIMARY payer (not Medicare).\n'
        '- Medicare does NOT cover overseas medical expenses.\n'
        '- Overseas claims require pay-up-front then submit within 3 years.\n'
        '- Medicare Part B must be actively enrolled (not automatic) for overseas residents.\n\n'
        'Answer questions based ONLY on the provided TRICARE documents below.\n'
        'If the answer is not in the documents, '
        'say "해당 내용은 제공된 문서에서 확인되지 않습니다." in the user\'s language.\n'
        'Always mention relevant conditions such as Group A/B, plan type, or beneficiary status.\n'
        '보험 추천이나 특정 플랜을 권유하는 답변은 하지 마세요.\n\n'
        # 출처 형식 고정: 형식만 학습, 실제 내용은 RAG가 채움
        '출처 표기 규칙:\n'
        '- PDF 출처: [출처]: 파일명(괄호 없이) 연도 p.N  예) [출처]: NGR_HB 2023 p.5\n'
        '- CSV 출처: [출처]: TRICARE https://tricare.mil/\n'
        '- 답변 마지막에 반드시 출처를 명시할 것\n\n'
        f'IMPORTANT LANGUAGE RULE:\n'
        f'- You must answer in {answer_language}.\n'
        f'- Use term locking: 본인부담금(Copay), 공제액(Deductible), 사전승인(Prior Authorization).\n\n'
        '[참고 문서]\n{context}\n\n'
        '[질문]\n{question}\n\n'
        '[답변]\n'
    )
    prompt = PromptTemplate.from_template(PROMPT_TEMPLATE)
    chain = (
        {'question': RunnablePassthrough(), 'context': ret | format_docs}
        | prompt
        | model
        | StrOutputParser()
    )
    return chain

print('✅ 함수 재정의 완료')

✅ 함수 재정의 완료


---
## Cell 1 · 질문 목록 정의

### 카테고리 비율
- 면책고지(일반 보험 정보): 35% → 140개
- 추천 방지: 20% → 80개
- 문맥: 10% → 40개
- 개인정보 차단: 35% → 140개

### 수량 줄이는 법
- 400→200개: 각 카테고리 리스트를 절반으로 슬라이싱
  예) `disclaimer_questions[:70]`, `reject_questions[:40]`, ...
- 400→100개: 각 카테고리를 1/4로
  예) `disclaimer_questions[:35]`, `reject_questions[:20]`, ...

In [4]:
# 카테고리별 질문 목록
# (question, type, category)
# type: 'rag' | 'table_rag' | 'reject_recommend' | 'reject_privacy'

# ── 면책 고지 (일반 보험 정보) — 140개 ──────────────────────────────
# Coverage, Cost, Eligibility, Pharmacy, OCONUS, Terminology 포함
# 답변 마지막에 면책 고지 + 출처 자동 삽입 학습 목적
disclaimer_questions = [
    # Coverage — 정신건강 (20개)
    ('TRICARE Prime에서 정신건강 상담(Psychotherapy)은 어떻게 보장되나요?', 'rag', 'disclaimer_coverage'),
    ('TRICARE Select에서 외래 정신건강 치료(Outpatient mental health)는 보장되나요?', 'rag', 'disclaimer_coverage'),
    ('TRICARE Prime에서 그룹 심리치료(Group therapy)도 커버되나요?', 'rag', 'disclaimer_coverage'),
    ('정신건강 입원 치료(Inpatient mental health)는 TRICARE에서 어떻게 보장되나요?', 'rag', 'disclaimer_coverage'),
    ('TRICARE에서 물질남용 치료(Substance use disorder treatment)는 어떻게 보장되나요?', 'rag', 'disclaimer_coverage'),
    ('Does TRICARE cover outpatient psychotherapy sessions?', 'rag', 'disclaimer_coverage'),
    ('How many mental health visits does TRICARE Prime cover per year?', 'rag', 'disclaimer_coverage'),
    ('TRICARE Prime에서 심리검사(Psychological testing)는 보장되나요?', 'rag', 'disclaimer_coverage'),
    ('TRICARE Select에서 가족 상담(Family therapy)은 보장 범위에 포함되나요?', 'rag', 'disclaimer_coverage'),
    ('Does TRICARE cover applied behavior analysis (ABA) therapy?', 'rag', 'disclaimer_coverage'),
    ('Is partial hospitalization for mental health covered by TRICARE?', 'rag', 'disclaimer_coverage'),
    ('TRICARE에서 중독 재활 프로그램(Residential treatment)은 보장되나요?', 'rag', 'disclaimer_coverage'),
    ('TRICARE Prime에서 심리상담 사전승인(Prior Authorization)이 필요한가요?', 'rag', 'disclaimer_coverage'),
    ('Does TRICARE require a referral for mental health outpatient visits?', 'rag', 'disclaimer_coverage'),
    ('TRICARE For Life에서 정신건강 치료는 Medicare와 어떻게 분담되나요?', 'rag', 'disclaimer_coverage'),
    ('한국에서 영어 심리상담을 TRICARE로 받을 수 있나요?', 'rag', 'disclaimer_coverage'),
    ('Is telehealth for mental health services covered under TRICARE?', 'rag', 'disclaimer_coverage'),
    ('TRICARE에서 심리상담사(Licensed Clinical Social Worker)에게 받는 치료도 보장되나요?', 'rag', 'disclaimer_coverage'),
    ('TRICARE에서 자살 예방 서비스(Suicide prevention)는 어떻게 지원되나요?', 'rag', 'disclaimer_coverage'),
    ('TRICARE에서 학습 장애(Learning disorders) 치료는 보장되나요?', 'rag', 'disclaimer_coverage'),

    # Coverage — 일반 의료 (20개)
    ('TRICARE Prime에서 예방 접종(Preventive vaccines)은 보장되나요?', 'rag', 'disclaimer_coverage'),
    ('TRICARE에서 임신·출산 관련 서비스는 어떻게 보장되나요?', 'rag', 'disclaimer_coverage'),
    ('Does TRICARE cover chiropractic care?', 'rag', 'disclaimer_coverage'),
    ('TRICARE에서 응급실(Emergency room) 방문은 어떻게 보장되나요?', 'rag', 'disclaimer_coverage'),
    ('TRICARE Select에서 전문의 진료(Specialty care) 시 사전승인이 필요한가요?', 'rag', 'disclaimer_coverage'),
    ('Is hearing aid coverage available under TRICARE?', 'rag', 'disclaimer_coverage'),
    ('Does TRICARE cover dental care for active duty family members?', 'rag', 'disclaimer_coverage'),
    ('TRICARE Prime에서 물리치료(Physical therapy)는 어떻게 보장되나요?', 'rag', 'disclaimer_coverage'),
    ('Is weight loss surgery covered by TRICARE?', 'rag', 'disclaimer_coverage'),
    ('Does TRICARE cover vision care and eyeglasses?', 'rag', 'disclaimer_coverage'),
    ('TRICARE에서 입원 수술(Inpatient surgery) 시 비용 분담은 어떻게 되나요?', 'rag', 'disclaimer_coverage'),
    ('TRICARE Prime에서 구급차(Ambulance) 이용은 보장되나요?', 'rag', 'disclaimer_coverage'),
    ('TRICARE에서 항암치료(Chemotherapy)는 어떻게 보장되나요?', 'rag', 'disclaimer_coverage'),
    ('Does TRICARE cover second opinions from civilian providers?', 'rag', 'disclaimer_coverage'),
    ('TRICARE에서 재활 치료(Rehabilitation services)는 어떻게 보장되나요?', 'rag', 'disclaimer_coverage'),
    ('Is experimental treatment covered by TRICARE?', 'rag', 'disclaimer_coverage'),
    ('Does TRICARE cover custodial care (long-term home assistance)?', 'rag', 'disclaimer_coverage'),
    ('Is marriage counseling covered by TRICARE?', 'rag', 'disclaimer_coverage'),
    ('Does TRICARE cover over-the-counter medications?', 'rag', 'disclaimer_coverage'),
    ('Does TRICARE cover gym membership or fitness programs?', 'rag', 'disclaimer_coverage'),

    # Cost — 비용 수치 (30개)
    ('TRICARE Prime Group A 현역 군인 가족의 외래 주치의 진료 본인부담금은 얼마인가요?', 'table_rag', 'disclaimer_cost'),
    ('TRICARE Select Group B 재향군인의 전문의 외래 진료비(네트워크 내)는 얼마인가요?', 'table_rag', 'disclaimer_cost'),
    ('TRICARE Prime에서 응급실 방문 시 Group A 재향군인의 본인부담금은 얼마인가요?', 'table_rag', 'disclaimer_cost'),
    ('TRICARE Reserve Select의 월 보험료(Member only)는 얼마인가요?', 'table_rag', 'disclaimer_cost'),
    ('TRICARE Prime Group A 재향군인의 연간 연납 보험료(Enrollment fee)는 얼마인가요?', 'table_rag', 'disclaimer_cost'),
    ('TRICARE Select Group A 현역 군인 가족의 연간 공제액(Deductible)은 얼마인가요?', 'table_rag', 'disclaimer_cost'),
    ('What is the catastrophic cap for TRICARE Prime Group B retirees?', 'table_rag', 'disclaimer_cost'),
    ('TRICARE Prime에서 입원(Inpatient surgery) 시 Group B 재향군인의 본인부담금은 얼마인가요?', 'table_rag', 'disclaimer_cost'),
    ('TRICARE Select의 외래 정신건강 치료 본인부담금은 얼마인가요?', 'table_rag', 'disclaimer_cost'),
    ('What is the annual enrollment fee for TRICARE Select Group A retirees?', 'table_rag', 'disclaimer_cost'),
    ('TRICARE For Life에서 Medicare Part B 공제액(Deductible)은 얼마인가요?', 'table_rag', 'disclaimer_cost'),
    ('TRICARE Prime에서 구급차 이용 시 본인부담금은 얼마인가요?', 'table_rag', 'disclaimer_cost'),
    ('TRICARE Prime Group A 재향군인의 외래 수술(Ambulatory surgery) 본인부담금은 얼마인가요?', 'table_rag', 'disclaimer_cost'),
    ('TRICARE Young Adult의 월 보험료는 얼마인가요?', 'table_rag', 'disclaimer_cost'),
    ('TRICARE Select Group B에서 입원 치료(Inpatient medical) 시 본인부담금은 얼마인가요?', 'table_rag', 'disclaimer_cost'),
    ('What is the deductible for TRICARE Select Group B retirees?', 'table_rag', 'disclaimer_cost'),
    ('TRICARE Prime에서 외래 정신건강 치료 본인부담금은 얼마인가요?', 'table_rag', 'disclaimer_cost'),
    ('TRICARE Reserve Select의 가족 포함 월 보험료는 얼마인가요?', 'table_rag', 'disclaimer_cost'),
    ('TRICARE Select에서 네트워크 외 전문의 진료 시 Group A 재향군인의 cost-share 비율은?', 'table_rag', 'disclaimer_cost'),
    ('What is the catastrophic cap for active duty family members under TRICARE?', 'table_rag', 'disclaimer_cost'),
    ('TRICARE Prime Group B 재향군인의 연간 보험료는 얼마인가요?', 'table_rag', 'disclaimer_cost'),
    ('TRICARE For Life에서 Medicare가 보장하는 서비스의 본인 부담은 얼마인가요?', 'table_rag', 'disclaimer_cost'),
    ('TRICARE Select Group A 재향군인의 외래 주치의 진료 본인부담금은 얼마인가요?', 'table_rag', 'disclaimer_cost'),
    ('TRICARE Prime에서 예방 진료(Preventive care) 본인부담금은 얼마인가요?', 'table_rag', 'disclaimer_cost'),
    ('What is the annual deductible for TRICARE Select Group A active duty family members?', 'table_rag', 'disclaimer_cost'),
    ('TRICARE Prime Group A 현역 군인 가족의 입원 치료 본인부담금은 얼마인가요?', 'table_rag', 'disclaimer_cost'),
    ('TRICARE Select Group B에서 구급차 이용 시 본인부담금은 얼마인가요?', 'table_rag', 'disclaimer_cost'),
    ('What is the annual catastrophic cap for TRICARE Select Group B retirees?', 'table_rag', 'disclaimer_cost'),
    ('What is the cost-share for TRICARE Prime Overseas beneficiaries in Korea?', 'table_rag', 'disclaimer_cost'),
    ('TRICARE Select Group A에서 입원 정신건강 치료 본인부담금은 얼마인가요?', 'table_rag', 'disclaimer_cost'),

    # Eligibility (20개)
    ('현역 군인 가족(Active Duty Family Member)은 어떤 TRICARE 플랜을 사용할 수 있나요?', 'rag', 'disclaimer_eligibility'),
    ('What TRICARE plans are available for retirees under age 65?', 'rag', 'disclaimer_eligibility'),
    ('TRICARE Reserve Select는 어떤 조건에서 가입할 수 있나요?', 'rag', 'disclaimer_eligibility'),
    ('국가방위군(National Guard)이 활성화(Activated)될 때 TRICARE 플랜은 어떻게 바뀌나요?', 'rag', 'disclaimer_eligibility'),
    ('TRICARE Young Adult은 누가 가입할 수 있나요?', 'rag', 'disclaimer_eligibility'),
    ('What is the age limit for dependents to be covered under TRICARE?', 'rag', 'disclaimer_eligibility'),
    ('65세가 되면 TRICARE에서 어떤 변화가 생기나요?', 'rag', 'disclaimer_eligibility'),
    ('TRICARE For Life 수혜 자격은 어떻게 되나요?', 'rag', 'disclaimer_eligibility'),
    ('Can divorced spouses remain on TRICARE coverage?', 'rag', 'disclaimer_eligibility'),
    ('TRICARE에서 전역 후(Separated from service) 30일 동안 보장이 어떻게 유지되나요?', 'rag', 'disclaimer_eligibility'),
    ('QLE(Qualifying Life Event)에 해당하는 사례는 어떤 것들이 있나요?', 'rag', 'disclaimer_eligibility'),
    ('Is a surviving spouse eligible for TRICARE coverage?', 'rag', 'disclaimer_eligibility'),
    ('한국에서 TRICARE Prime Overseas에 등록하려면 어떻게 해야 하나요?', 'rag', 'disclaimer_eligibility'),
    ('What happens to TRICARE coverage when a service member deploys?', 'rag', 'disclaimer_eligibility'),
    ('Can a retiree enroll in TRICARE Prime Overseas while living in Korea?', 'rag', 'disclaimer_eligibility'),
    ('TRICARE에서 Medicare Part B 등록이 의무인 경우는 언제인가요?', 'rag', 'disclaimer_eligibility'),
    ('국가방위군이 비활성화 상태일 때 사용할 수 있는 TRICARE 플랜은 무엇인가요?', 'rag', 'disclaimer_eligibility'),
    ('TRICARE 적격성 확인은 어디서 할 수 있나요?', 'rag', 'disclaimer_eligibility'),
    ('TRICARE Prime에서 다른 플랜으로 변경하려면 어떤 절차가 필요한가요?', 'rag', 'disclaimer_eligibility'),
    ('What are the eligibility requirements for TRICARE Retired Reserve?', 'rag', 'disclaimer_eligibility'),

    # OCONUS (25개)
    ('한국(South Korea)에서 TRICARE를 사용할 수 있나요?', 'rag', 'disclaimer_oconus'),
    ('TRICARE Prime Overseas와 TRICARE Select Overseas의 차이는 무엇인가요?', 'rag', 'disclaimer_oconus'),
    ('한국에서 TRICARE 청구(Claim) 시 선불 후 환급 방식으로 해야 하나요?', 'rag', 'disclaimer_oconus'),
    ('What is the TRICARE Overseas Program (TOP) and how does it work in Korea?', 'rag', 'disclaimer_oconus'),
    ('한국에서 TRICARE를 사용할 때 Medicare와의 우선순위는 어떻게 되나요?', 'rag', 'disclaimer_oconus'),
    ('OCONUS에서 응급 의료를 받은 경우 TRICARE 청구는 어떻게 하나요?', 'rag', 'disclaimer_oconus'),
    ('TRICARE Overseas에서 청구 제출 기한(Deadline)은 얼마나 되나요?', 'rag', 'disclaimer_oconus'),
    ('Does Medicare cover medical expenses incurred in South Korea?', 'rag', 'disclaimer_oconus'),
    ('USFK 관련 수혜자가 한국에서 이용할 수 있는 TRICARE 옵션은 무엇인가요?', 'rag', 'disclaimer_oconus'),
    ('한국에서 TRICARE Prime Overseas 플랜 등록 방법은 무엇인가요?', 'rag', 'disclaimer_oconus'),
    ('What is the TRICARE Near Patient Program and is it available in Korea?', 'rag', 'disclaimer_oconus'),
    ('TRICARE Overseas에서 처방약은 어떻게 수령할 수 있나요?', 'rag', 'disclaimer_oconus'),
    ('한국에서 TRICARE Prime Remote Overseas를 사용할 수 있나요?', 'rag', 'disclaimer_oconus'),
    ('TRICARE Overseas에서 정신건강 치료는 어떻게 받을 수 있나요?', 'rag', 'disclaimer_oconus'),
    ('한국에서 응급상황 발생 시 TRICARE 사용 절차는 어떻게 되나요?', 'rag', 'disclaimer_oconus'),
    ('TRICARE Select Overseas에서 한국 현지 병원 이용 시 비용 분담은 어떻게 되나요?', 'rag', 'disclaimer_oconus'),
    ('Does TRICARE Prime Overseas require a referral for specialist care in Korea?', 'rag', 'disclaimer_oconus'),
    ('TRICARE Overseas에서 보장이 되지 않는 해외 서비스가 있나요?', 'rag', 'disclaimer_oconus'),
    ('TRICARE Overseas 청구서 제출은 어디에 해야 하나요?', 'rag', 'disclaimer_oconus'),
    ('What documents are needed for a TRICARE overseas claim submission?', 'rag', 'disclaimer_oconus'),
    ('한국에서 출산(Maternity care)을 한 경우 TRICARE 보장은 어떻게 되나요?', 'rag', 'disclaimer_oconus'),
    ('TRICARE Overseas에서 치과 서비스는 보장되나요?', 'rag', 'disclaimer_oconus'),
    ('한국에서 미국 군병원(MTF)과 민간 병원 이용 시 TRICARE 비용 차이는?', 'rag', 'disclaimer_oconus'),
    ('한국에서 TRICARE 네트워크 병원을 찾으려면 어떻게 해야 하나요?', 'rag', 'disclaimer_oconus'),
    ('TRICARE Overseas에서 사전승인(Prior Authorization)이 필요한 경우는 언제인가요?', 'rag', 'disclaimer_oconus'),

    # Pharmacy (15개)
    ('TRICARE 약국 프로그램(Pharmacy Program)에서 처방약은 어떻게 받을 수 있나요?', 'rag', 'disclaimer_pharmacy'),
    ('TRICARE Home Delivery Pharmacy를 이용하려면 어떻게 등록하나요?', 'rag', 'disclaimer_pharmacy'),
    ('What is the cost for generic drugs at a TRICARE retail pharmacy?', 'rag', 'disclaimer_pharmacy'),
    ('한국에서 TRICARE로 처방약을 받을 수 있나요?', 'rag', 'disclaimer_pharmacy'),
    ('TRICARE Formulary에 없는 약은 어떻게 처방받나요?', 'rag', 'disclaimer_pharmacy'),
    ('TRICARE 약국 프로그램에서 브랜드 약과 제네릭 약의 비용 차이는 얼마인가요?', 'rag', 'disclaimer_pharmacy'),
    ('What is TRICARE Mail Order Pharmacy and how do I use it overseas?', 'rag', 'disclaimer_pharmacy'),
    ('TRICARE 약국 프로그램에서 90일 처방과 30일 처방의 비용 차이는 무엇인가요?', 'rag', 'disclaimer_pharmacy'),
    ('Can I use a non-network pharmacy for TRICARE and get reimbursed?', 'rag', 'disclaimer_pharmacy'),
    ('TRICARE에서 처방약 우선승인(Prior Authorization for medications)이 필요한 경우는 언제인가요?', 'rag', 'disclaimer_pharmacy'),
    ('What are the TRICARE drug tiers and how do they affect cost?', 'rag', 'disclaimer_pharmacy'),
    ('TRICARE에서 군 의무시설(MTF) 약국 이용 시 비용은 얼마인가요?', 'rag', 'disclaimer_pharmacy'),
    ('한국에서 TRICARE 약국 프로그램 이용이 가능한가요?', 'rag', 'disclaimer_pharmacy'),
    ('TRICARE에서 전문 의약품(Specialty drugs)은 어떻게 처방받나요?', 'rag', 'disclaimer_pharmacy'),
    ('TRICARE에서 정신건강 치료약(Psychiatric medications)은 어떻게 보장되나요?', 'rag', 'disclaimer_pharmacy'),

    # Terminology (10개)
    ('TRICARE에서 Deductible(공제액)이 무엇인가요?', 'rag', 'disclaimer_terminology'),
    ('What is a catastrophic cap in TRICARE?', 'rag', 'disclaimer_terminology'),
    ('TRICARE에서 Prior Authorization(사전승인)은 어떤 절차인가요?', 'rag', 'disclaimer_terminology'),
    ('Group A와 Group B의 차이는 무엇인가요?', 'rag', 'disclaimer_terminology'),
    ('What does OCONUS mean and why does it matter for TRICARE?', 'rag', 'disclaimer_terminology'),
    ('TRICARE에서 Enrollment fee(등록비)와 Deductible은 다른 건가요?', 'rag', 'disclaimer_terminology'),
    ('What is the difference between TRICARE Prime and TRICARE Select?', 'rag', 'disclaimer_terminology'),
    ('TRICARE에서 MTF(Military Treatment Facility)란 무엇인가요?', 'rag', 'disclaimer_terminology'),
    ('What is DEERS and why do I need to be registered?', 'rag', 'disclaimer_terminology'),
    ('TRICARE에서 Transitional Survivor란 누구를 가리키나요?', 'rag', 'disclaimer_terminology'),
]

# ── 추천 방지 — 80개 ─────────────────────────────────────────────
# 플랜 추천 요청 → 정보 비교만 제공, 추천/권유 거절
reject_recommend_questions = [
    ('TRICARE Prime과 TRICARE Select 중 어떤 게 더 나은가요?', 'reject_recommend', 'reject_recommend'),
    ('저한테 제일 좋은 TRICARE 플랜을 추천해주세요.', 'reject_recommend', 'reject_recommend'),
    ('Which TRICARE plan should I choose for mental health coverage?', 'reject_recommend', 'reject_recommend'),
    ('TRICARE Reserve Select와 TRICARE Young Adult 중 어느 것이 더 유리한가요?', 'reject_recommend', 'reject_recommend'),
    ('한국에 사는데 어떤 TRICARE 플랜이 가장 적합한지 추천해주세요.', 'reject_recommend', 'reject_recommend'),
    ('비용 측면에서 TRICARE Prime이 Select보다 낫나요?', 'reject_recommend', 'reject_recommend'),
    ('For mental health in Korea, which plan is the best TRICARE option?', 'reject_recommend', 'reject_recommend'),
    ('가족이 있는데 어떤 TRICARE 플랜이 제일 좋나요?', 'reject_recommend', 'reject_recommend'),
    ('I want the cheapest TRICARE option. Which do you recommend?', 'reject_recommend', 'reject_recommend'),
    ('TRICARE 플랜 중 정신건강 커버리지가 제일 좋은 게 어떤 건가요?', 'reject_recommend', 'reject_recommend'),
    ('주한미군인데 어떤 TRICARE 플랜 써야 해요?', 'reject_recommend', 'reject_recommend'),
    ('은퇴 후 한국에 살 건데 TRICARE 어떤 걸 선택해야 하나요?', 'reject_recommend', 'reject_recommend'),
    ('Which TRICARE plan covers the most overseas services?', 'reject_recommend', 'reject_recommend'),
    ('약 많이 먹는데 어떤 TRICARE 플랜이 유리해요?', 'reject_recommend', 'reject_recommend'),
    ('TRICARE For Life랑 TRICARE Prime 중 뭐가 더 좋아요?', 'reject_recommend', 'reject_recommend'),
    ('예비군인데 어떤 플랜이 제일 저렴하게 이용 가능한가요?', 'reject_recommend', 'reject_recommend'),
    ('Should I switch from TRICARE Prime to TRICARE Select?', 'reject_recommend', 'reject_recommend'),
    ('임신 중인데 어떤 TRICARE 플랜이 좋을까요?', 'reject_recommend', 'reject_recommend'),
    ('TRICARE Select Overseas가 Prime Overseas보다 나은 점이 있나요?', 'reject_recommend', 'reject_recommend'),
    ('What TRICARE plan is best for a family with young children in Korea?', 'reject_recommend', 'reject_recommend'),
    ('노인인데 TRICARE For Life 써야 하나요 아니면 다른 거 써야 하나요?', 'reject_recommend', 'reject_recommend'),
    ('Is TRICARE Prime Overseas worth the enrollment fee?', 'reject_recommend', 'reject_recommend'),
    ('치과 치료 많이 받는데 어떤 TRICARE 플랜이 유리한가요?', 'reject_recommend', 'reject_recommend'),
    ('Which plan should I pick if I want the lowest out-of-pocket costs?', 'reject_recommend', 'reject_recommend'),
    ('입원 걱정되는데 TRICARE Prime이 낫나요 Select가 낫나요?', 'reject_recommend', 'reject_recommend'),
    ('TRICARE 처음 써보는데 어떤 플랜부터 알아보면 될까요?', 'reject_recommend', 'reject_recommend'),
    ('What is the most cost-effective TRICARE plan for retirees living overseas?', 'reject_recommend', 'reject_recommend'),
    ('TRICARE Young Adult이랑 TRICARE Select 중 뭐가 더 싼가요?', 'reject_recommend', 'reject_recommend'),
    ('I need the best mental health coverage. Which plan do you suggest?', 'reject_recommend', 'reject_recommend'),
    ('한국 거주자한테 TRICARE Overseas 옵션 중 뭐가 제일 좋아요?', 'reject_recommend', 'reject_recommend'),
    ('응급 치료 대비하려면 어떤 플랜이 좋아요?', 'reject_recommend', 'reject_recommend'),
    ('Which TRICARE plan is best for active duty members in Korea?', 'reject_recommend', 'reject_recommend'),
    ('TRICARE 연간 비용이 가장 적게 드는 플랜이 뭐예요?', 'reject_recommend', 'reject_recommend'),
    ('가입비 없는 TRICARE 플랜 있나요? 그게 더 좋은 건가요?', 'reject_recommend', 'reject_recommend'),
    ('Should I enroll in TRICARE Prime or just go with Select overseas?', 'reject_recommend', 'reject_recommend'),
    ('처방약 많이 먹는데 TRICARE 약국 프로그램 쓰려면 어떤 플랜이 유리해요?', 'reject_recommend', 'reject_recommend'),
    ('Is TRICARE Select better than Prime for retirees?', 'reject_recommend', 'reject_recommend'),
    ('한국에서 분만할 예정인데 어떤 TRICARE 플랜이 유리한가요?', 'reject_recommend', 'reject_recommend'),
    ('TRICARE 플랜 중 정신과 상담 비용이 가장 적게 나오는 거 어떤 거예요?', 'reject_recommend', 'reject_recommend'),
    ('Which TRICARE plan has the best pharmacy benefits?', 'reject_recommend', 'reject_recommend'),
    ('64세인데 내년에 65세 되면 TRICARE 플랜 바꿔야 하나요?', 'reject_recommend', 'reject_recommend'),
    ('Can you recommend the best TRICARE option for a disabled veteran?', 'reject_recommend', 'reject_recommend'),
    ('TRICARE Select Overseas 가입할 가치 있나요?', 'reject_recommend', 'reject_recommend'),
    ('어떤 플랜이 해외 청구 절차가 제일 간단해요?', 'reject_recommend', 'reject_recommend'),
    ('What TRICARE plan covers orthodontic treatment for my child?', 'reject_recommend', 'reject_recommend'),
    ('재향군인인데 TRICARE Prime이 좋을까요 Select가 좋을까요?', 'reject_recommend', 'reject_recommend'),
    ('Which plan is best if I move back to the US from Korea?', 'reject_recommend', 'reject_recommend'),
    ('TRICARE TRS랑 TRICARE Select 중에 뭐가 나아요?', 'reject_recommend', 'reject_recommend'),
    ('우리 아이 치아 교정 커버되는 플랜 추천해주세요.', 'reject_recommend', 'reject_recommend'),
    ('I want comprehensive mental health coverage. Which plan is best?', 'reject_recommend', 'reject_recommend'),
    ('현역 전역 예정인데 TRICARE 어떻게 유지하면 좋을까요?', 'reject_recommend', 'reject_recommend'),
    ('What is the best TRICARE plan for National Guard members?', 'reject_recommend', 'reject_recommend'),
    ('약값 아끼려면 어떤 TRICARE 플랜 써야 해요?', 'reject_recommend', 'reject_recommend'),
    ('Which TRICARE plan gives the most benefits for the lowest premium?', 'reject_recommend', 'reject_recommend'),
    ('한국에서 정신건강 치료 많이 받을 것 같은데 어떤 플랜이 나을까요?', 'reject_recommend', 'reject_recommend'),
    ('Should my spouse enroll in TRICARE Prime or Select?', 'reject_recommend', 'reject_recommend'),
    ('TRICARE에서 가장 포괄적인 해외 보장을 받으려면 어떤 플랜 선택해야 해요?', 'reject_recommend', 'reject_recommend'),
    ('Is there a TRICARE plan that covers everything with no copay?', 'reject_recommend', 'reject_recommend'),
    ('재향군인 Group B인데 어떤 플랜이 제일 저렴한가요?', 'reject_recommend', 'reject_recommend'),
    ('Which plan minimizes out-of-pocket for inpatient mental health in Korea?', 'reject_recommend', 'reject_recommend'),
    ('TRICARE 처음 가입하는 사람한테 어떤 플랜 알려줘요?', 'reject_recommend', 'reject_recommend'),
    ('I need surgery in Korea. Which TRICARE plan is best?', 'reject_recommend', 'reject_recommend'),
    ('새 군 가족인데 어떤 TRICARE 플랜 들어야 해요?', 'reject_recommend', 'reject_recommend'),
    ('TRICARE Prime Remote Overseas가 뭔지 모르는데 나한테 맞는지 알려줄 수 있어요?', 'reject_recommend', 'reject_recommend'),
    ('Which plan should retirees use if they only visit the US once a year?', 'reject_recommend', 'reject_recommend'),
    ('전직 군인인데 TRICARE 옵션 중 최선이 뭔지 알려주세요.', 'reject_recommend', 'reject_recommend'),
    ('Can you help me pick the best plan based on my health needs?', 'reject_recommend', 'reject_recommend'),
    ('Which TRICARE plan is better for families living in OCONUS locations?', 'reject_recommend', 'reject_recommend'),
    ('TRICARE Young Adult 아들한테 어떤 플랜 추천해줄 수 있어요?', 'reject_recommend', 'reject_recommend'),
    ('입원 잦은 편인데 어떤 플랜이 본인부담 제일 낮아요?', 'reject_recommend', 'reject_recommend'),
    ('What TRICARE plan has the lowest enrollment fee for retirees?', 'reject_recommend', 'reject_recommend'),
    ('한국 거주 미군 가족인데 제일 좋은 TRICARE 옵션 알려주세요.', 'reject_recommend', 'reject_recommend'),
    ('TRICARE 비교해서 제일 좋은 플랜 골라줘요.', 'reject_recommend', 'reject_recommend'),
    ('Is TRICARE Prime or Select better for mental health outpatient visits?', 'reject_recommend', 'reject_recommend'),
    ('TRICARE 가입 전 제일 유리한 플랜이 뭔지 알고 싶어요.', 'reject_recommend', 'reject_recommend'),
    ('국가방위군인데 어떤 TRICARE 플랜 써야 해요?', 'reject_recommend', 'reject_recommend'),
    ('Which plan covers the most services for USFK families in Korea?', 'reject_recommend', 'reject_recommend'),
]

# ── 문맥 이해 — 40개 ─────────────────────────────────────────────
# 복합적이거나 문맥 파악이 필요한 질문
context_questions = [
    ('TRICARE Prime에서 먼저 주치의(PCM) 진료 후 전문의에게 가야 하는데, 한국에서는 이게 어떻게 적용되나요?', 'rag', 'context'),
    ('저는 65세가 넘었고 Medicare도 있는데, 한국에서 수술하면 TRICARE For Life와 Medicare 중 어느 것이 먼저 적용되나요?', 'rag', 'context'),
    ('전역 후 60일이 지났는데 아직 새 보험을 못 구했어요. TRICARE를 계속 쓸 수 있나요?', 'rag', 'context'),
    ('국가방위군인데 갑자기 해외 파병이 결정됐습니다. 제 TRICARE 보장은 어떻게 바뀌나요?', 'rag', 'context'),
    ('TRICARE Select Overseas로 한국 병원에서 치료받고 청구했는데 일부만 환급됐어요. 왜 그런 건가요?', 'rag', 'context'),
    ('주한미군 가족인데 응급으로 한국 민간 병원 갔어요. 사전승인 없이 갔는데 환급받을 수 있나요?', 'rag', 'context'),
    ('TRICARE 공제액(Deductible)을 이미 채웠는데 이후 청구는 어떻게 되나요?', 'rag', 'context'),
    ('한국에서 TRICARE로 정신건강 치료를 받으려는데 한국어로 진료하는 제공자도 보장이 되나요?', 'rag', 'context'),
    ('TRICARE Prime에서 PCM 변경을 요청했는데 거절됐어요. 그러면 다른 의사한테 가면 어떻게 되나요?', 'rag', 'context'),
    ('Catastrophic Cap에 도달했는데 남은 기간 동안 비용이 어떻게 되나요?', 'rag', 'context'),
    ('TRICARE 보장 중에 사전승인이 필요한 항목인지 어떻게 알 수 있나요?', 'rag', 'context'),
    ('한국에서 TRICARE로 받은 치료인데 미국 달러로 환급받는 건가요 원화로 받는 건가요?', 'rag', 'context'),
    ('제 플랜은 TRICARE Select인데, 네트워크 외 병원을 이용할 경우 공제액이 따로 적용되나요?', 'rag', 'context'),
    ('TRICARE 청구 시 제출 기한 3년이 지나면 어떻게 되나요?', 'rag', 'context'),
    ('TRICARE For Life 가입자인데 한국에서 치료받으면 Medicare Secondary Payer 규칙이 적용되나요?', 'rag', 'context'),
    ('I was treated at a Korean hospital but the doctor was not a TRICARE-authorized provider. Can I still get reimbursed?', 'rag', 'context'),
    ('TRICARE Prime Overseas에서 전문의 진료에 항상 의뢰서(referral)가 필요한가요, 아니면 예외도 있나요?', 'rag', 'context'),
    ('한국에서 물리치료를 10회 받았는데 TRICARE에서 다 커버해주나요?', 'rag', 'context'),
    ('My child needs ABA therapy while living in Korea. How does TRICARE cover this overseas?', 'rag', 'context'),
    ('예비군에서 활성화됐다가 다시 비활성화됐어요. 플랜 전환은 자동인가요?', 'rag', 'context'),
    ('TRICARE 약국 프로그램에서 한국 현지 약국은 네트워크 약국인가요?', 'rag', 'context'),
    ('한국에서 출산했는데 신생아도 바로 TRICARE 보장을 받을 수 있나요?', 'rag', 'context'),
    ('TRICARE Select에서 연간 공제액을 채우기 전에 응급 치료를 받으면 어떻게 되나요?', 'rag', 'context'),
    ('TRICARE 청구서에 진단 코드가 없으면 어떻게 되나요?', 'rag', 'context'),
    ('두 가지 보험(TRICARE + 다른 민간보험)이 있는데 한국에서 치료받으면 어떻게 조율되나요?', 'rag', 'context'),
    ('My TRICARE claim was denied. What should I mention in my appeal?', 'rag', 'context'),
    ('TRICARE Prime 플랜에서 입원 전 사전승인 받는 절차가 한국에서도 동일한가요?', 'rag', 'context'),
    ('한국 KATUSA 복무 중인데 TRICARE 보장이 어떻게 되나요?', 'rag', 'context'),
    ('TRICARE Select Overseas의 cost-share가 CONUS 버전과 다른가요?', 'rag', 'context'),
    ('TRICARE Young Adult는 부모님의 보험에 얹혀 있는 건가요 별도 가입인가요?', 'rag', 'context'),
    ('이혼 후에도 전 배우자가 TRICARE를 쓸 수 있는 조건이 뭔가요?', 'rag', 'context'),
    ('Can a Korean national working on a US base receive TRICARE benefits?', 'rag', 'context'),
    ('TRICARE에서 치료 후 청구 시 영문 의료 기록이 꼭 필요한가요, 한국어도 되나요?', 'rag', 'context'),
    ('TRICARE Prime Overseas에서 의뢰 없이 응급실 간 경우와 의뢰 받고 간 경우 본인부담 차이가 있나요?', 'rag', 'context'),
    ('한국에서 약을 받았는데 처방전이 한국어로만 되어 있어요. TRICARE 청구에 문제가 없나요?', 'rag', 'context'),
    ('TRICARE 청구 시 EOB(Explanation of Benefits)는 어디서 받나요?', 'rag', 'context'),
    ('한국에서 TRICARE 사용 중에 미국으로 일시 귀국했을 때 보장이 어떻게 되나요?', 'rag', 'context'),
    ('TRICARE 가입자가 사망한 경우 배우자의 보장은 어떻게 유지되나요?', 'rag', 'context'),
    ('My TRICARE claim was partially paid. How do I understand the breakdown?', 'rag', 'context'),
    ('TRICARE에서 "medically necessary"의 기준은 무엇인가요?', 'rag', 'context'),
]

# ── 개인정보 차단 — 140개 ────────────────────────────────────────
# 식별 가능한 개인정보(보험번호, SSN, 청구번호 등) 포함 → 차단
# 진단명·증상만 있는 경우 → 차단 X, 일반 답변
privacy_block_questions = [
    # 차단 O: 식별 가능 정보 포함 (70개)
    ('제 보험번호 123-45-6789이고요, PTSD 진단을 받았는데 TRICARE에서 치료 커버해줄까요?', 'reject_privacy', 'reject_privacy'),
    ('I have a diagnosis of F32.1 (Major Depression). My SSN is 123-45-6789. Will TRICARE cover my treatment?', 'reject_privacy', 'reject_privacy'),
    ('제 주민등록번호와 보험 카드 번호를 알려드릴 테니 커버리지를 확인해주세요.', 'reject_privacy', 'reject_privacy'),
    ('My SSN is 123-45-6789. Can you check if I qualify for TRICARE?', 'reject_privacy', 'reject_privacy'),
    ('제 개인 TRICARE 청구 내역을 조회할 수 있나요?', 'reject_privacy', 'reject_privacy'),
    ('Can you look up my TRICARE claim history using my sponsor SSN?', 'reject_privacy', 'reject_privacy'),
    ('제 ICD 코드가 F41.1이고 보험번호는 TC-98765입니다. 이 진단 처리가 가능한가요?', 'reject_privacy', 'reject_privacy'),
    ('My sponsor ID is 987-65-4321. Can you verify my TRICARE eligibility?', 'reject_privacy', 'reject_privacy'),
    ('청구 번호 CLM-2024-001234로 제출한 건 언제 처리되나요?', 'reject_privacy', 'reject_privacy'),
    ('제 DEERS 등록 번호 알려드릴게요. 보장 내역 확인해주세요.', 'reject_privacy', 'reject_privacy'),
    ('My claim ID is TC123456. Can you check the status?', 'reject_privacy', 'reject_privacy'),
    ('보험카드 번호 4521-XXXX-XXXX-1234로 조회해주세요.', 'reject_privacy', 'reject_privacy'),
    ('제 군번 1234567이고 PTSD 치료 받으려는데 커버되나요?', 'reject_privacy', 'reject_privacy'),
    ('My DoD ID is 1234567890 and I need mental health coverage confirmation.', 'reject_privacy', 'reject_privacy'),
    ('생년월일과 보험번호 알려드릴게요 — 1990년 1월 1일, TC-11111. 보장 확인해주세요.', 'reject_privacy', 'reject_privacy'),
    ('내 계좌번호 123-456-78910이에요. 환급금 여기로 보내주세요.', 'reject_privacy', 'reject_privacy'),
    ('Can you process my reimbursement to routing number 021000021, account 12345678?', 'reject_privacy', 'reject_privacy'),
    ('제 주소는 서울시 용산구이고 군번은 9876543입니다. 자격 확인해주세요.', 'reject_privacy', 'reject_privacy'),
    ('My patient ID at the MTF is P-00456. Can you look up my records?', 'reject_privacy', 'reject_privacy'),
    ('보험번호 TC-77777과 진단명 F20(조현병)으로 사전승인 가능한지 확인해주세요.', 'reject_privacy', 'reject_privacy'),
    ('제 SSN 마지막 4자리는 5678이에요. 이걸로 자격 확인돼요?', 'reject_privacy', 'reject_privacy'),
    ('I want to provide my sponsor SSN to check my dependent coverage status.', 'reject_privacy', 'reject_privacy'),
    ('청구 레퍼런스 번호 REF-2024-KOR-00789 진행 상황 알려주세요.', 'reject_privacy', 'reject_privacy'),
    ('제 TRICARE 회원 ID로 지난 청구 내역 조회해줄 수 있나요?', 'reject_privacy', 'reject_privacy'),
    ('My TRICARE member number is TM-123456. Please confirm my benefits.', 'reject_privacy', 'reject_privacy'),
    ('이름이 홍길동이고 생년월일 1985-03-15, 보험번호 TC-99999예요. 혜택 확인해줘요.', 'reject_privacy', 'reject_privacy'),
    ('My full name is John Smith, SSN 555-44-3333. Am I covered for overseas?', 'reject_privacy', 'reject_privacy'),
    ('군번이랑 보험 카드 번호 알려드릴 테니까 커버리지 상세히 알려주세요.', 'reject_privacy', 'reject_privacy'),
    ('Can I give you my DoD ID and have you verify my TRICARE plan details?', 'reject_privacy', 'reject_privacy'),
    ('제 진료 기록 번호 MR-2024-001로 보장 여부 확인해주세요.', 'reject_privacy', 'reject_privacy'),
    ('보험번호 TC-55555이고 불안장애 진단이에요. 이 조합으로 커버 확인돼요?', 'reject_privacy', 'reject_privacy'),
    ('My claim was rejected. My claim number is CLM-98765. Can you help?', 'reject_privacy', 'reject_privacy'),
    ('제 이메일이랑 보험번호 드릴게요 — kjw@example.com, TC-00001. 확인해주세요.', 'reject_privacy', 'reject_privacy'),
    ('스폰서 SSN으로 제 수혜자 자격을 조회해줄 수 있나요?', 'reject_privacy', 'reject_privacy'),
    ('My authorization number is AUTH-2024-KOR-123. Is it still valid?', 'reject_privacy', 'reject_privacy'),
    ('제 군 이메일 주소로 보장 내용 보내줄 수 있어요?', 'reject_privacy', 'reject_privacy'),
    ('제 보험 가입일이랑 보험번호 알려드릴게요. 지금도 유효한지 확인해주세요.', 'reject_privacy', 'reject_privacy'),
    ('Can you verify coverage using my member ID TC-77788 and date of birth?', 'reject_privacy', 'reject_privacy'),
    ('보험번호랑 전화번호 드릴게요 — TC-44444, 010-1234-5678. 커버리지 알려주세요.', 'reject_privacy', 'reject_privacy'),
    ('My SSN and name: Jane Doe, 999-88-7777. Confirm TRICARE For Life benefits.', 'reject_privacy', 'reject_privacy'),
    ('병원 영수증 번호 INV-2024-KR-001이에요. 이걸로 청구 조회 가능한가요?', 'reject_privacy', 'reject_privacy'),
    ('제 진단 코드 F33.0이고 보험번호 TC-11223입니다. 사전승인 처리해주세요.', 'reject_privacy', 'reject_privacy'),
    ('My TRICARE beneficiary ID is BID-2024-0001. Can you confirm eligibility?', 'reject_privacy', 'reject_privacy'),
    ('군번이랑 가족관계증명서 번호 알려드릴게요. 수혜자 등록 확인해줘요.', 'reject_privacy', 'reject_privacy'),
    ('I was hospitalized last month. Claim ref: CLM-2024-789. Any update?', 'reject_privacy', 'reject_privacy'),
    ('제 스폰서 DoD ID가 0987654321이에요. 가족 보장 확인 부탁해요.', 'reject_privacy', 'reject_privacy'),
    ('보험사 측에 제 케이스 번호 CASE-2024-KR-9999를 조회해달라고 할 수 있나요?', 'reject_privacy', 'reject_privacy'),
    ('My insurance member number is MEM-2024-56789. Check my plan details.', 'reject_privacy', 'reject_privacy'),
    ('제 청구 내역 확인하려면 보험번호 드리면 돼요? TC-22334입니다.', 'reject_privacy', 'reject_privacy'),
    ('Can you look up my overseas claim CLM-OCONUS-2024-001 status?', 'reject_privacy', 'reject_privacy'),
    ('이름이랑 생년월일, 보험번호 제공하면 보장 내역 다 알 수 있나요?', 'reject_privacy', 'reject_privacy'),
    ('My TRICARE enrollment confirmation number is ENRL-2024-00123. Is it active?', 'reject_privacy', 'reject_privacy'),
    ('보험번호 TC-99001이고 진단 코드 F84.0이에요. 치료 커버 여부 확인해주세요.', 'reject_privacy', 'reject_privacy'),
    ('Can you verify if member ID TC-55544 is still enrolled in Prime Overseas?', 'reject_privacy', 'reject_privacy'),
    ('제 케이스 매니저한테 연락해달라고 할 수 있어요? 제 번호는 010-9999-8888이에요.', 'reject_privacy', 'reject_privacy'),
    ('My EOB reference number is EOB-2024-KR-7654. Can you explain the charges?', 'reject_privacy', 'reject_privacy'),
    ('스폰서 이름이랑 SSN 알려드릴게요 — Mark Johnson, 111-22-3333.', 'reject_privacy', 'reject_privacy'),
    ('제 TRICARE 청구 상태를 회원 ID TC-88990으로 확인해주세요.', 'reject_privacy', 'reject_privacy'),
    ('보험번호 TC-12345로 제 입원 청구가 승인됐는지 알려주세요.', 'reject_privacy', 'reject_privacy'),
    ("My DoD ID is 1122334455 and I'm trying to confirm my TRICARE Select enrollment.", 'reject_privacy', 'reject_privacy'),
    ('제 청구 레퍼런스 CLM-KOR-2024-5555로 처리 현황 알 수 있나요?', 'reject_privacy', 'reject_privacy'),
    ('Can you confirm if my claim AUTH-KOR-2024-777 was pre-authorized?', 'reject_privacy', 'reject_privacy'),
    ('제 이름, 생년월일, 보험번호 드릴게요. 전체 보장 내역 출력해주세요.', 'reject_privacy', 'reject_privacy'),
    ("I'd like to provide my SSN and member ID for a coverage verification.", 'reject_privacy', 'reject_privacy'),
    ('보험번호랑 주소 드릴게요. 제 청구 처리 현황 알 수 있나요?', 'reject_privacy', 'reject_privacy'),
    ('My patient number at the clinic is PN-2024-00987. Can you pull my benefits?', 'reject_privacy', 'reject_privacy'),
    ('제 TRICARE 계정 비밀번호 재설정 좀 해주세요. 이메일은 abc@example.com이에요.', 'reject_privacy', 'reject_privacy'),
    ('Can you help me log into my TRICARE account? My username is jsmith1985.', 'reject_privacy', 'reject_privacy'),
    ('보험번호 TC-00777이고 주민번호 앞 6자리는 850315예요. 확인해줘요.', 'reject_privacy', 'reject_privacy'),
    ('My claim was submitted with reference CLM-OCONUS-9999. When will it be paid?', 'reject_privacy', 'reject_privacy'),

    # 차단 X: 진단명·증상만 있고 식별 정보 없음 → 일반 정보 답변 (70개)
    ('저는 우울증인데 TRICARE에서 정신건강 치료를 받을 수 있나요?', 'rag', 'context_health_info'),
    ('내가 PTSD 진단을 받았는데 TRICARE에서 커버되나요?', 'rag', 'context_health_info'),
    ('불안장애가 있는데 TRICARE에서 상담 치료 받을 수 있나요?', 'rag', 'context_health_info'),
    ('I have bipolar disorder. Does TRICARE cover my medication?', 'rag', 'context_health_info'),
    ('조현병 진단인데 TRICARE 입원 치료 커버되나요?', 'rag', 'context_health_info'),
    ('I was diagnosed with major depressive disorder. What mental health benefits do I have?', 'rag', 'context_health_info'),
    ('아이가 자폐 진단을 받았는데 TRICARE에서 ABA 치료 커버해주나요?', 'rag', 'context_health_info'),
    ('My child has ADHD. Does TRICARE cover behavioral therapy?', 'rag', 'context_health_info'),
    ('알코올 의존증인데 TRICARE에서 치료 프로그램 보장이 되나요?', 'rag', 'context_health_info'),
    ('I struggle with substance use. What TRICARE benefits apply?', 'rag', 'context_health_info'),
    ('섭식 장애 있는데 TRICARE에서 치료 커버 돼요?', 'rag', 'context_health_info'),
    ('My teenager has an eating disorder. Is residential treatment covered?', 'rag', 'context_health_info'),
    ('공황장애 있는데 TRICARE 약 처방 커버되나요?', 'rag', 'context_health_info'),
    ('I have generalized anxiety disorder. How does TRICARE cover therapy?', 'rag', 'context_health_info'),
    ('OCD 치료약이 TRICARE Formulary에 있나요?', 'rag', 'context_health_info'),
    ('I was recently diagnosed with schizophrenia. What TRICARE benefits apply?', 'rag', 'context_health_info'),
    ('산후 우울증인데 TRICARE에서 상담 치료 보장이 돼요?', 'rag', 'context_health_info'),
    ('My spouse has PTSD from deployment. What mental health services are covered?', 'rag', 'context_health_info'),
    ('수면 장애 치료 받고 싶은데 TRICARE에서 커버되나요?', 'rag', 'context_health_info'),
    ('I have chronic back pain. What does TRICARE cover for physical therapy?', 'rag', 'context_health_info'),
    ('당뇨 진단인데 TRICARE에서 인슐린 처방 커버되나요?', 'rag', 'context_health_info'),
    ('I have type 2 diabetes. Are my medications covered under TRICARE?', 'rag', 'context_health_info'),
    ('고혈압 약 TRICARE에서 커버돼요?', 'rag', 'context_health_info'),
    ('I was recently diagnosed with hypertension. What pharmacy benefits apply?', 'rag', 'context_health_info'),
    ('암 진단을 받았어요. TRICARE에서 항암 치료 커버가 어떻게 되나요?', 'rag', 'context_health_info'),
    ('I have been diagnosed with breast cancer. What TRICARE benefits do I have?', 'rag', 'context_health_info'),
    ('심장 수술이 필요한데 TRICARE에서 커버되나요?', 'rag', 'context_health_info'),
    ('I need knee replacement surgery. How does TRICARE cover this?', 'rag', 'context_health_info'),
    ('만성 신장 질환인데 투석 치료 TRICARE에서 커버되나요?', 'rag', 'context_health_info'),
    ('I have kidney disease and need dialysis. Is this covered under TRICARE?', 'rag', 'context_health_info'),
    ('임신 중인데 산전 검사는 TRICARE에서 무료인가요?', 'rag', 'context_health_info'),
    ("I'm pregnant. What maternity services does TRICARE cover?", 'rag', 'context_health_info'),
    ('자궁내막증 치료 TRICARE에서 보장되나요?', 'rag', 'context_health_info'),
    ('I was diagnosed with endometriosis. What treatments does TRICARE cover?', 'rag', 'context_health_info'),
    ('천식 있는데 TRICARE에서 흡입기 처방 커버돼요?', 'rag', 'context_health_info'),
    ('My child has asthma. Are nebulizer treatments covered by TRICARE?', 'rag', 'context_health_info'),
    ('뇌졸중 후 재활 치료 TRICARE에서 보장돼요?', 'rag', 'context_health_info'),
    ('I had a stroke. What rehabilitation services are covered under TRICARE?', 'rag', 'context_health_info'),
    ('허리 디스크 진단인데 TRICARE에서 MRI 커버되나요?', 'rag', 'context_health_info'),
    ('I have a herniated disc. Does TRICARE cover MRI and specialist visits?', 'rag', 'context_health_info'),
    ('알레르기 면역 치료 TRICARE에서 보장돼요?', 'rag', 'context_health_info'),
    ('I have severe allergies. Does TRICARE cover allergy shots?', 'rag', 'context_health_info'),
    ('갑상선 질환 치료 TRICARE에서 커버되나요?', 'rag', 'context_health_info'),
    ('I was diagnosed with hypothyroidism. Are my medications covered?', 'rag', 'context_health_info'),
    ('류마티스 관절염 치료 약 TRICARE에서 커버돼요?', 'rag', 'context_health_info'),
    ('I have rheumatoid arthritis. What specialty drugs does TRICARE cover?', 'rag', 'context_health_info'),
    ('청력 손실이 있는데 TRICARE에서 보청기 커버되나요?', 'rag', 'context_health_info'),
    ('I have hearing loss. Does TRICARE cover hearing aids?', 'rag', 'context_health_info'),
    ('시력 교정 수술 TRICARE에서 보장되나요?', 'rag', 'context_health_info'),
    ('I need LASIK surgery. Is it covered by TRICARE?', 'rag', 'context_health_info'),
    ('수술 후 물리치료 TRICARE에서 몇 회까지 커버돼요?', 'rag', 'context_health_info'),
    ('I need post-surgery physical therapy. How many sessions does TRICARE cover?', 'rag', 'context_health_info'),
    ('다발성 경화증(MS) 치료 TRICARE에서 보장돼요?', 'rag', 'context_health_info'),
    ('I have multiple sclerosis. What does TRICARE cover for my condition?', 'rag', 'context_health_info'),
    ('파킨슨병 진단인데 TRICARE에서 어떤 치료가 보장되나요?', 'rag', 'context_health_info'),
    ('I was diagnosed with Parkinson\'s disease. What TRICARE benefits apply?', 'rag', 'context_health_info'),
    ('척추 측만증 교정 수술 TRICARE에서 커버되나요?', 'rag', 'context_health_info'),
    ('My child has scoliosis. Does TRICARE cover corrective surgery?', 'rag', 'context_health_info'),
    ('치매 진단인데 TRICARE에서 어떤 지원이 되나요?', 'rag', 'context_health_info'),
    ('My parent has dementia. What TRICARE services are available?', 'rag', 'context_health_info'),
    ('간질(뇌전증) 치료 TRICARE에서 커버돼요?', 'rag', 'context_health_info'),
    ('I have epilepsy. Are anti-seizure medications covered under TRICARE?', 'rag', 'context_health_info'),
    ('크론병 치료 TRICARE에서 보장돼요?', 'rag', 'context_health_info'),
    ('I have Crohn\'s disease. What treatments does TRICARE cover?', 'rag', 'context_health_info'),
    ('루푸스 진단인데 TRICARE에서 어떤 치료가 커버되나요?', 'rag', 'context_health_info'),
    ('I have lupus. What TRICARE benefits apply to autoimmune treatment?', 'rag', 'context_health_info'),
    ('HIV 양성인데 TRICARE에서 항바이러스 약 커버되나요?', 'rag', 'context_health_info'),
    ('I am HIV positive. Does TRICARE cover antiretroviral therapy?', 'rag', 'context_health_info'),
    ('대장암 진단인데 TRICARE에서 수술이랑 항암 치료 커버되나요?', 'rag', 'context_health_info'),
    ('I have colon cancer. What does TRICARE cover for chemotherapy?', 'rag', 'context_health_info'),
    ('수술 후 감염으로 재입원했는데 TRICARE에서 추가 입원도 커버되나요?', 'rag', 'context_health_info'),
]

# 전체 질문 목록 조합
# 수량 조절 시 각 리스트 뒤에 슬라이싱 추가
# 예: 400→200개로 줄이려면:
#   QUESTIONS = disclaimer_questions[:70] + reject_recommend_questions[:40] + context_questions[:20] + privacy_block_questions[:70]
# 400개 생성: QUESTIONS = disclaimer_questions + reject_recommend_questions + context_questions + privacy_block_questions
QUESTIONS = disclaimer_questions[:35] + reject_recommend_questions[:20] + context_questions[:10] + privacy_block_questions[:35]

print(f'총 질문 수: {len(QUESTIONS)}개')
from collections import Counter
cats = Counter(cat for _, _, cat in QUESTIONS)
for cat, cnt in sorted(cats.items()):
    print(f'  {cat}: {cnt}개')

총 질문 수: 100개
  context: 10개
  disclaimer_coverage: 35개
  reject_privacy: 35개
  reject_recommend: 20개


---
## Cell 2 · 거절 패턴 템플릿 정의

In [5]:
# 거절 패턴 템플릿

REJECT_RECOMMEND_KO = (
    "본 서비스는 특정 TRICARE 플랜을 추천하거나 가입을 권유하지 않습니다.\n\n"
    "각 플랜의 보장 범위(Coverage), 본인부담금(Copay), 공제액(Deductible), "
    "등록비(Enrollment Fee) 등 객관적인 조건을 비교해 드릴 수 있습니다. "
    "어떤 플랜들의 조건을 비교해 드릴까요?\n\n"
    "⚠️ 최종 플랜 선택은 본인의 상황과 수혜 자격을 고려하여 "
    "TRICARE 공식 담당자(tricare.mil) 또는 공인 보험 설계사와 직접 상담하시기 바랍니다."
)

REJECT_RECOMMEND_EN = (
    "This service does not recommend or endorse any specific TRICARE plan.\n\n"
    "I can provide objective information on coverage, copay, deductible, and enrollment fees "
    "for each plan so you can make an informed decision. "
    "Which plans would you like to compare?\n\n"
    "⚠️ For final plan selection, please consult directly with a TRICARE representative "
    "at tricare.mil or a licensed benefits counselor."
)

# 개인 식별 정보 포함 시 차단 (보험번호, SSN, 청구번호 등)
REJECT_PRIVACY_KO = (
    "개인 식별 정보(보험번호, 군번, 청구번호, 주민등록번호, 계좌번호 등)는 "
    "이 서비스에서 필요하지 않으며 저장되지 않습니다.\n\n"
    "본 서비스는 공개된 TRICARE 문서를 기반으로 플랜 레벨의 보장 정보를 안내합니다. "
    "이용 중인 TRICARE 플랜명을 알려주시면, 해당 플랜의 일반적인 보장 범위(Coverage)와 "
    "본인부담금(Copay) 기준을 안내해 드릴 수 있습니다.\n\n"
    "⚠️ 개인 계약별 적용 여부는 반드시 TRICARE 공식 채널(tricare.mil)을 통해 확인하세요."
)

REJECT_PRIVACY_EN = (
    "Personal identifying information such as insurance IDs, SSNs, claim numbers, "
    "or account numbers are not required and are not stored by this service.\n\n"
    "This service provides plan-level coverage information based on official TRICARE documents. "
    "Please share the name of your TRICARE plan, and I can provide general coverage "
    "and cost-share information for that plan.\n\n"
    "⚠️ For individual claim eligibility, please contact TRICARE directly at tricare.mil."
)

def get_reject_answer(question: str, reject_type: str) -> str:
    lang = detect_language(question)
    if reject_type == 'reject_recommend':
        return REJECT_RECOMMEND_KO if lang == 'ko' else REJECT_RECOMMEND_EN
    elif reject_type == 'reject_privacy':
        return REJECT_PRIVACY_KO if lang == 'ko' else REJECT_PRIVACY_EN
    return '해당 질문에 답변할 수 없습니다.'

print('✅ 거절 패턴 템플릿 정의 완료')

✅ 거절 패턴 템플릿 정의 완료


---
## Cell 3 · 데이터셋 생성 (메인 루프)

In [6]:
import json
import time

SYSTEM_PROMPT = (
    "당신은 TRICARE 군 건강보험 전문 안내 assistant입니다. "
    "주한미군(USFK) 및 한국 거주 수혜자를 포함한 OCONUS 수혜자를 주요 대상으로 합니다. "
    "공개된 TRICARE 공식 문서 기반으로만 플랜 정보를 제공하며, "
    "특정 플랜 추천·가입 권유는 하지 않습니다. "
    "개인 식별 정보(보험번호·SSN·청구번호 등)는 요청하거나 저장하지 않습니다. "
    "모든 PDF 출처는 '[출처]: 파일명 연도 p.N' 형식으로, "
    "CSV 출처는 '[출처]: TRICARE https://tricare.mil/' 형식으로 명시합니다. "
    "보험 전문 용어는 한국어 답변 시 원어 병기 형식(예: 본인부담금(Copay))으로 표기합니다."
)

dataset = []
errors  = []

for idx, (question, q_type, category) in enumerate(QUESTIONS):
    try:
        if q_type in ('reject_recommend', 'reject_privacy'):
            answer = get_reject_answer(question, q_type)
            # 수정: 거절 패턴은 RAG 검색 없이 템플릿 답변만 사용하므로 sources를 빈 리스트로 초기화
            sources = []

        elif q_type == 'table_rag':
            table_chain = make_rag_chain(question, table_retriever, model)
            answer = table_chain.invoke(question)
            # 수정: table_retriever로 검색된 문서에서 source_file, page, location 메타데이터를 수집
            docs = table_retriever.invoke(question)
            sources = [
                {
                    'source_file': d.metadata.get('source_file', d.metadata.get('source', 'unknown')),
                    'page':        d.metadata.get('page', ''),
                    'location':    d.metadata.get('location', ''),
                }
                for d in docs
            ]

        else:  # rag
            chain = make_rag_chain(question, retriever, model)
            answer = chain.invoke(question)
            # 수정: retriever로 검색된 문서에서 source_file, page, location 메타데이터를 수집
            # 평가지표 생성 시 정답 청크 ID로 활용하기 위해 별도 저장
            docs = retriever.invoke(question)
            sources = [
                {
                    'source_file': d.metadata.get('source_file', d.metadata.get('source', 'unknown')),
                    'page':        d.metadata.get('page', ''),
                    'location':    d.metadata.get('location', ''),
                }
                for d in docs
            ]

        dataset.append({
            'messages': [
                {'role': 'system',    'content': SYSTEM_PROMPT},
                {'role': 'user',      'content': question},
                {'role': 'assistant', 'content': answer},
            ],
            'metadata': {
                'category': category,
                'type':     q_type,
                # 수정: sources 필드 추가 — 검색된 문서의 출처 정보를 메타데이터에 함께 저장
                # 나중에 평가지표(hit rate, Recall@k) 계산 시 정답 청크 식별에 활용
                'sources':  sources,
            }
        })

        if (idx + 1) % 10 == 0:
            print(f'[{idx+1}/{len(QUESTIONS)}] 완료 — 최근 질문: {question[:40]}...')

        time.sleep(0.5)  # API rate limit 방지

    except Exception as e:
        errors.append({'idx': idx, 'question': question, 'error': str(e)})
        print(f'  ⚠️ [{idx}] 오류: {question[:40]} → {e}')
        continue

print(f'\n✅ 생성 완료: {len(dataset)}개 / 오류: {len(errors)}개')

[10/100] 완료 — 최근 질문: Does TRICARE cover applied behavior anal...
[20/100] 완료 — 최근 질문: TRICARE에서 학습 장애(Learning disorders) 치료는 ...
[30/100] 완료 — 최근 질문: Does TRICARE cover vision care and eyegl...
[40/100] 완료 — 최근 질문: 한국에 사는데 어떤 TRICARE 플랜이 가장 적합한지 추천해주세요....
[50/100] 완료 — 최근 질문: TRICARE For Life랑 TRICARE Prime 중 뭐가 더 좋...
[60/100] 완료 — 최근 질문: TRICARE Select Overseas로 한국 병원에서 치료받고 청구...
[70/100] 완료 — 최근 질문: 제 개인 TRICARE 청구 내역을 조회할 수 있나요?...
[80/100] 완료 — 최근 질문: 생년월일과 보험번호 알려드릴게요 — 1990년 1월 1일, TC-1111...
[90/100] 완료 — 최근 질문: My TRICARE member number is TM-123456. P...
[100/100] 완료 — 최근 질문: My authorization number is AUTH-2024-KOR...

✅ 생성 완료: 100개 / 오류: 0개


---
## Cell 4 · JSONL 저장

In [7]:
OUTPUT_DIR = './'

# 파인튜닝 메인 파일 (messages만, 학습 시 사용)
main_path = OUTPUT_DIR + 'tricare_finetuning_dataset.jsonl'
with open(main_path, 'w', encoding='utf-8') as f:
    for item in dataset:
        f.write(json.dumps({'messages': item['messages']}, ensure_ascii=False) + '\n')

# 수정: 메타데이터 포함 버전 저장 추가
# messages + metadata(category, type, sources) 를 모두 포함한 파일
# 검증·평가지표 계산 시 사용 (학습에는 사용하지 않음)
meta_path = OUTPUT_DIR + 'tricare_finetuning_dataset_with_meta.jsonl'
with open(meta_path, 'w', encoding='utf-8') as f:
    for item in dataset:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

# 오류 목록
if errors:
    err_path = OUTPUT_DIR + 'tricare_finetuning_errors.json'
    with open(err_path, 'w', encoding='utf-8') as f:
        json.dump(errors, f, ensure_ascii=False, indent=2)
    print(f'⚠️ 오류 목록 저장: {err_path}')

print(f'✅ 저장 완료')
print(f'   메인:     {main_path} ({len(dataset)}개)')
# 수정: meta_path 저장 확인 출력 추가
print(f'   메타포함: {meta_path}')

✅ 저장 완료
   메인:     ./tricare_finetuning_dataset.jsonl (100개)
   메타포함: ./tricare_finetuning_dataset_with_meta.jsonl


---
## Cell 5 · 데이터셋 검증

In [8]:
import json
from collections import Counter

# 수정: 검증 대상을 main_path(messages만) 대신 meta_path(메타데이터 포함)로 변경
# sources 필드 포함 여부를 함께 확인하기 위함
loaded = []
with open(meta_path, 'r', encoding='utf-8') as f:
    for line in f:
        loaded.append(json.loads(line.strip()))

print(f'총 데이터 수: {len(loaded)}개\n')

# 카테고리 분포
cats = Counter(item['metadata']['category'] for item in loaded)
print('── 카테고리 분포 ──')
for cat, cnt in sorted(cats.items(), key=lambda x: -x[1]):
    print(f'  {cat}: {cnt}개')

# 답변 길이 분포
lengths = [len(item['messages'][2]['content']) for item in loaded]
print(f'\n── 답변 길이 분포 ──')
print(f'  최소: {min(lengths)}자')
print(f'  최대: {max(lengths)}자')
print(f'  평균: {sum(lengths)//len(lengths)}자')

# 짧은 답변 확인 (150자 미만 → 출처 미검색 가능성)
short = [(item['messages'][1]['content'], len(item['messages'][2]['content']))
         for item in loaded if len(item['messages'][2]['content']) < 150]
if short:
    print(f'\n⚠️ 짧은 답변 ({len(short)}개) — 재생성 권장:')
    for q, l in short:
        print(f'  [{l}자] {q[:60]}')
else:
    print('\n✅ 모든 답변 길이 정상')

# 출처 형식 확인
source_ok = sum(1 for item in loaded if '[출처]' in item['messages'][2]['content'])
print(f'\n출처 포함 답변: {source_ok}/{len(loaded)}개')

# 수정: sources 필드 수집 여부 확인 추가
# reject 타입은 sources=[] 이므로 rag/table_rag 타입 기준으로 집계
has_sources = sum(
    1 for item in loaded
    if item['metadata'].get('sources') and len(item['metadata']['sources']) > 0
)
print(f'sources 수집된 항목: {has_sources}/{len(loaded)}개')

# 샘플 출력 (metadata 포함)
print('\n── 샘플 (첫 번째) ──')
sample = loaded[0]
print(f'Q: {sample["messages"][1]["content"]}')
print(f'A: {sample["messages"][2]["content"][:200]}...')
# 수정: 샘플 출력에 metadata(sources 포함) 추가
print(f'Meta: {sample["metadata"]}')

총 데이터 수: 100개

── 카테고리 분포 ──
  disclaimer_coverage: 35개
  reject_privacy: 35개
  reject_recommend: 20개
  context: 10개

── 답변 길이 분포 ──
  최소: 61자
  최대: 614자
  평균: 284자

⚠️ 짧은 답변 (9개) — 재생성 권장:
  [61자] Does TRICARE cover applied behavior analysis (ABA) therapy?
  [61자] Is partial hospitalization for mental health covered by TRIC
  [61자] TRICARE에서 중독 재활 프로그램(Residential treatment)은 보장되나요?
  [61자] TRICARE For Life에서 정신건강 치료는 Medicare와 어떻게 분담되나요?
  [61자] Is telehealth for mental health services covered under TRICA
  [61자] TRICARE에서 자살 예방 서비스(Suicide prevention)는 어떻게 지원되나요?
  [61자] TRICARE에서 학습 장애(Learning disorders) 치료는 보장되나요?
  [61자] Is hearing aid coverage available under TRICARE?
  [61자] Is weight loss surgery covered by TRICARE?

출처 포함 답변: 45/100개
sources 수집된 항목: 0/100개

── 샘플 (첫 번째) ──
Q: TRICARE Prime에서 정신건강 상담(Psychotherapy)은 어떻게 보장되나요?
A: TRICARE Prime에서 정신건강 상담(Psychotherapy)은 다음과 같이 보장됩니다. 정신건강 상담은 사전승인(Prior Authorization)이 필요하며, 상담을 받기 위해서는 TRICARE 네트워크 내의 제공자에게 가야 합니다. 본인부담금(Copa